# Rapid ML App Deployment - From Model to Application

<div style="text-align:center;">
<img src="../images/stock/pexels-googledeepmind-17483874.jpg" alt="Google Deepmind Generated Image">
</div>
This notebook provides a streamlined workflow for transitioning your Machine Learning models from Jupyter Notebooks into a functional, interactive application. We will focus on the critical "last mile" of data science: model serialization and user interface development using `Joblib` and `Streamlit`.


## Exporting Models

To save your Jupyter Notebook trained machine learning model for future use, we use the `joblib` library to `dump` the object into a persistent file:

```python
import joblib

joblib.dump(your_model, "custom_filename.joblib")
```

And that's all there is to it! Your model is now stored safely on your hard drive.


Breakdown of the Code:

* `joblib.dump()`: The primary function used to serialize a Python object (like your model) and write it to a file on your disk.
* `your_model`: The variable name of your trained model object.
* `custom_filename.joblib`: The destination path and name for your file. While you can use any extension, `.joblib` is the standard convention to indicate the file was created using this library.

__Note__: If your model is particularly large, you can add `compress=3` inside the `dump` function to reduce the file size without sacrificing much speed.

### Important Note: Security and Best Practices
When working with `joblib` and pickle, there is one "Golden Rule" of security you must follow to keep your system safe.

#### The Security Risk
Because `joblib.load()` essentially "reconstructs" a Python object from a file, a malicious actor could create a fake model file that contains harmful code. If you load that file, that code will execute on your computer immediately.

#### The Golden Rule of Model Loading
* Never load a `.joblib` or `.pkl` file from an untrusted source.

* Only load files that you created yourself.

* Only load files from trusted teammates or verified, secure repositories.

* Never download a "pre-trained model" from an anonymous link on the internet and run `joblib.load()` on it without verifying its origin.

## Developing the ML Application with `Streamlit`

<div style="text-align:center;">
    <img src="../images/web/streamlit-logo-secondary-colormark-darktext.png" alt="Streamlit Logo" width=600>
</div>

Now that our model is ready, we will use `Streamlit` to transform it into an interactive web application. `Streamlit` allows us to data tools using pure Python—no HTML or JavaScript required.

We will follow these four core steps to get the app running:

1. __Environment Setup__: Create a dedicated virtual environment to manage dependencies and avoid library conflicts.

2. __Dependency Installation__: Install `Streamlit`, `Joblib`, and any libraries (like `Scikit-Learn` or `Pandas`) required to run your model.

3. __Application Development__: Write the Python script that defines the user interface, handles inputs, and generates predictions.

4. __Deployment & Execution__: Launch the `Streamlit` server locally to view and test your live application in the browser.

Before we start, make sure your exported model file is in the same folder where you plan to write your `Streamlit` code.

## Setting Up Your Virtual Environment

Before we write any code, we need a clean workspace. Open your Terminal (or the integrated terminal in VSCode) and create a new directory for your project.

Once you are inside that folder, run the following command to initialize your virtual environment:

```bash
python -m venv venv
```

### Command Breakdown:
* `python`: This calls the Python interpreter installed on your system.

* `-m venv`: This tells Python to run the built-in "Virtual Environment" module.

* `venv`: This is the name of the folder that will be created to house your isolated libraries. While you can technically name it anything, `venv` is the industry standard.

## Activating the Virtual Environment

Once the environment is created, you must activate it. This tells your terminal to use the isolated Python instance and libraries located inside your venv folder rather than the global ones on your computer.

### For Mac/Linux:

```bash
source venv/bin/activate
```

For Windows:
```bash
.\venv\Scripts\activate
```

### Command Breakdown:
* `source` (Mac/Linux): A shell command that executes the script in the current environment context.

* `.\` (Windows): Tells the terminal to look in the current directory for the subsequent path.

* `venv/bin/` or `venv\Scripts\`: The specific folder where the activation scripts are stored by Python.

* `activate`: The script that actually modifies your terminal's "PATH" variable to point to your new virtual environment.

__Quick Check__: You’ll know it worked if you see (`venv`) appear in parentheses at the start of your terminal command prompt.

### Installing the Necessary Packages

Now that your virtual environment is active, you need to install the libraries that will power your app. Run the following command in your terminal:

```bash
pip install pandas scikit-learn streamlit joblib
```

Package Breakdown:
* `pip install`: The standard package manager command used to download and install Python libraries from the `PyPI` repository.

* `pandas`: The essential tool for data manipulation and cleaning. It handles the dataframes your model expects.

* `scikit-learn`: The machine learning library used to build and define the structure of your model.

* `streamlit`: The framework that turns your Python script into a functional web application with sliders, buttons, and charts.

* `joblib`: The utility we use to "unfreeze" (load) your saved model file so it can make real-time predictions.

__Note:__ After installing the packages, it’s a great habit to run `pip freeze > requirements.txt`. This creates a list of your packages so others can recreate your environment.

## Building the App

### Building the App - Importing Libraries

Building the App: Importing Libraries
The first step in our Python script is to import the tools we just installed.

```python
import pandas as pd
import streamlit as st
import numpy as np
import joblib
```

### Building the App - Loading the Model

Now we need to bring your saved model into the application. We use a "cached" function to ensure the model is loaded only once, which keeps the app snappy and efficient.

```python
# 1. Define the filename
file = 'your_model.joblib'

# 2. Define a cached function to load the model
@st.cache_resource
def load_model(file):
    model = joblib.load(file)
    return model

# 3. Safely execute the loading process
try:
    model = load_model(file)

except FileNotFoundError:
    st.error("Error: Model File Not Found")
    st.stop()
```

#### Code Breakdown:
* `file = 'your_model.joblib'`: A variable storing the exact name of your exported model. Ensure this file is in the same directory as your script.

* `@st.cache_resource`: A `Streamlit` "decorator." It tells the app: "Load this once and keep it in memory." Without this, `Streamlit` would reload the massive model file every time a user clicks a button, making the app very slow.

* `joblib.load(file_path)`: The inverse of `dump`. it reads the file from your disk and converts it back into a functional Python object (your model).

* `try` / `except`: A safety net. If the file is missing, the except block catches the error and displays a user-friendly message instead of crashing the whole program.

* `st.error()` & `st.stop()`: Displays a red error box in the browser and halts the app's execution until the issue is fixed.



### Building the App - The User Interface 

<div style="text-align:center;">
    <img src="../images/web/pokemon_toys.jpg" alt="A Pokeball surround by Pokemon Toys" width=400>
</div>
<br>
<br>

In this section, we use the `Streamlit` layout tools to create a clean, two-column input form. This specific example is designed for a Pokemon Type Predictor.

```python
# Page Header
st.title("Pokemon Type Predictor")
st.write("Enter Stats Below")

# Two Column Layout for Inputs
col1, col2 = st.columns(2)

with col1:
	hp = st.number_input("HP", min_value=1, value=50)
	attack = st.number_input("Attack", min_value=1, value=60)
	defense = st.number_input("Defense", min_value=1, value=60)
	sp_atk = st.number_input("Special Attack", min_value=1, value=60)

with col2:
	sp_def = st.number_input("Special Defense", min_value=1, value=60)
	speed = st.number_input("Speed", min_value=1, value=90)
	generation = st.number_input("Generation", min_value=1, max_value=5, value=1)
	legendary = st.toggle("Legendary")
```

#### UI Breakdown:

* `st.title()` & `st.write()`: These set the tone of your app. The title is rendered as an `<h1>` tag, and write handles standard descriptive text.

* `st.columns(2)`: This creates a side-by-side layout. Using `with col1:` ensures that all the code indented below it stays on the left side of the screen.

* `st.number_input()`: This widget creates a numeric field. We use `min_value` to prevent users from entering invalid data (like 0 or negative HP) and `value` to set a sensible starting point.

* `st.toggle()`: A modern UI switch used for Boolean inputs. Perfect for categorical flags like "Legendary."

### Building the App - Data Processing and Prediction Logic

``` python
# Feature Engineering
total = hp + attack + defense + sp_atk + sp_def + speed

# Reshaping for the Model
features = np.array([[hp, attack, defense, sp_atk, sp_def, speed, total, legendary, generation]])

# Predictive Trigger
if st.button("Predict Type"):
	prediction = model.predict(features)

	st.divider()

    # Result Display
	if prediction[0] == True:
		st.success("Electric")
		st.balloons()
	else:
		st.warning("Not Electric")
```

#### Logic Breakdown:
* `total = ...`: Since the model uses a "Total Stats" feature, we calculate this manually by summing all the individual base stats.

* `np.array([[...]])`: Most Scikit-Learn models expect a 2D array (a list of lists). The double brackets `[[ ]]` ensure the model sees one row of data with multiple columns.

* `st.button()`: This acts as a logical gate. The prediction code only runs once the user confirms their inputs by clicking the button, preventing the app from guessing prematurely.

* `prediction[0]`: The `.predict()` method usually returns a list or array of results. Since we only sent one Pokemon's stats, we look at the first item at index 0.

* `st.success()` & `st.warning()`: These provide color-coded feedback (green for positive, yellow/red for negative) to make the user experience intuitive.

* `st.balloons()`: A built-in `Streamlit` animation that adds a touch of "wit" and celebration to a successful prediction.

### Running the App

With your code written and your virtual environment active, it’s time to see your creation in action. Use the following command in your terminal to launch the `Streamlit` server:

```bash
streamlit run main.py
```

#### Command Breakdown:
* `streamlit run`: This tells the `Streamlit` library to initialize a local web server. Instead of just running the script once and stopping (like a normal Python file), it keeps the script "alive" and watches for changes.

* `main.py`: This is the name of your Python file. If you named your file something else, make sure to use that name instead.

#### What Happens Next?
* __Local URL__: Your terminal will display a "Local URL" (usually __http://localhost:8501__).

* __Browser Launch__: Your default web browser should automatically open to that address.

* __Hot Reloading__: One of the best features of __Streamlit__ is that if you save changes to your __main.py__ file, you can simply click "Always rerun" in the browser to see your updates instantly without restarting the terminal.

Congratulations. You’ve successfully built a Machine Learning web application from scratch.

### Stopping the `Streamlit` App (The Server)
To stop the app from running so you can type new commands in the terminal, you use a "Keyboard Interrupt":

__Command:__ `Ctrl + C`

__Result:__ This deactivates the local web server. The website will still be open in the browser, but it will show a "Connection Error" because the the terminal process has been turned off.

### Deactivating the Virtual Environment
The deactivate command is specifically for you are using a virtual environment (like `venv` or `conda`).

__Command:__ `deactivate`

__Result:__ This doesn't stop the app, it just "exits" the special folder where the Python libraries are installed.

## Want to Learn More?

The journey doesn't end here. To explore more advanced features like interactive charts, sidebar navigation, or deploying your app to the cloud, check out the official resources:

* [Streamlit Official Website](https://streamlit.io/)

* [Streamlit Documentation](https://docs.streamlit.io/)

* [Streamlit Cheat Sheet](https://docs.streamlit.io/develop/quick-reference/cheat-sheet)